# Axis 4 — Compute / I-O Concurrency & Pipeline Overlap (Windows dGPU / PyTorch)

**Original question (team page).** Can dense *shared-expert* computation fully mask the latency of transferring routed experts?

**Reframed for OLMoE** (as on the Mac). OLMoE has **no shared expert**, so once a layer's router has chosen its 8 experts, the work that *can* run while that layer's missing experts are being loaded is:

1. the GEMMs of the experts that were **hits** (already in VRAM), and
2. **other loads** — several misses can be read in parallel (NVMe queue depth), each copied to VRAM on its own CUDA stream.

The next layer's attention cannot start early: it needs this layer's output. (Using the *previous* layer's time to load — prefetching — is Axis 7.)

**Hypothesis, restated.** Hit-expert compute hides a meaningful part of one miss's latency, but when *m* experts miss at once the load time exceeds the compute window and the stall is exposed, growing with *m*.

## Modes compared (all real, same requests, same cache decisions)
| Mode | What happens after routing is known |
|---|---|
| `sync-1` | load misses one after another (disk → pinned buffer → VRAM), then launch all expert GEMMs |
| `sync-4` | load misses on 4 reader threads in parallel (each with its own staging buffer and copy stream), then compute |
| `overlap-1` | launch the hit-expert GEMMs on the GPU first (CUDA kernels are asynchronous), load misses on 1 thread meanwhile, then the missed experts |
| `overlap-4` | same with 4 reader threads |

**Exact in every mode (differs from the Mac).** Each expert's weighted output is computed separately and the outputs are summed in the stock expert order at the end, so `overlap` changes only *when* work runs, never the result: every mode must produce identical tokens and identical miss patterns (asserted). The Mac's `overlap` added two partial sums and was not bit-exact.

**Per MoE call** we time `T_moe` = routing known → layer output synchronised, and group it by **m = number of misses** in the call. Exposed stall(m) = T_moe(m) − T_moe(0). A load here is disk read **plus** host → VRAM copy. `MOE_SMOKE=1` runs a tiny configuration.

In [ ]:
import os, sys, time, json, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Shared code: the `moe` package in src/. `pip install -e ".[windows]"` installs it; the path
# line below makes the notebook work without installing.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'moe').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moe import common, torch_cache as tc, results as rs, figures as fg   # tc is Windows (PyTorch)
for m in (common, tc, rs, fg):
    importlib.reload(m)                # pick up edits to src/moe without restarting the kernel

# MOE_SMOKE=1 runs a tiny configuration end to end (used to test the notebook).
SMOKE = os.environ.get('MOE_SMOKE') == '1'
AXIS_DIR = ROOT / 'experiments' / 'axis-4-io-compute-overlap'
RESULTS = (ROOT / 'experiments' / '.cache' / 'smoke' / 'axis-4-io-compute-overlap') if SMOKE else (AXIS_DIR / 'results')
RESULTS.mkdir(parents=True, exist_ok=True)

DEVICE = 'windows-dgpu'                 # results file: <axis>/results/windows-dgpu.json
AXIS = 'axis-4-io-compute-overlap'
TABLES = {}                             # every table this run produces, saved at the end

GPU = tc.default_device()               # the dedicated GPU (Windows + CUDA only); K counts experts in its VRAM
torch.set_grad_enabled(False)

plt.rcParams.update({'figure.figsize': (11, 5), 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})
print('SMOKE run' if SMOKE else 'Full run', '| results ->', RESULTS.relative_to(ROOT))
print(tc.system_info(GPU))

## Configuration

In [ ]:
K = 256
MODES = {'sync-1': ('sync', 1), 'sync-4': ('sync', 4), 'overlap-1': ('overlap', 1), 'overlap-4': ('overlap', 4)}
REQUESTS_PER_DOMAIN = 1          # x 4 kinds of request (max 2)
MAX_NEW_TOKENS = 48
MICRO_THREADS = [1, 2, 4, 8]
MICRO_REPS = 12

if SMOKE:
    REQUESTS_PER_DOMAIN, MAX_NEW_TOKENS, MICRO_THREADS, MICRO_REPS = 1, 8, [1, 4], 3
print('K =', K, '| modes:', list(MODES))
CONFIG = rs.config_from(globals())   # recorded in the results file

## Model (non-expert weights only) and correctness gate

In [ ]:
MODEL_DIR = tc.default_model_dir()       # allenai/OLMoE-1B-7B-0125-Instruct (BF16), downloaded beforehand
assert list(MODEL_DIR.glob('model*.safetensors')),     f'Checkpoint not found in {MODEL_DIR}: download allenai/OLMoE-1B-7B-0125-Instruct there first.'
model, tokenizer, index = tc.load_model(MODEL_DIR, device=GPU)
print(index.describe())
print(f'Non-expert weights resident on {GPU}: {tc.non_expert_bytes(model) / 1e9:.2f} GB '
      f'| expert reads: {tc.ExpertReader(index).mode}')
# Measured ONCE, before any expert bank exists: free VRAM read later in the run is unreliable.
SAFE_K = tc.max_safe_capacity(index, GPU)
print(f'Largest K that fits in the VRAM free right now: {SAFE_K} (bank {SAFE_K * index.expert_bytes / 1e9:.1f} GB)')

# Correctness gate: one layer through the K-slot bank vs the stock transformers layer
# (all 64 experts of that layer resident, eager expert loop). Must be bit-identical.
diff = tc.check_layer_exactness(model, index, capacity=64)
assert diff == 0.0, diff
print(f'Layer exactness vs stock transformers: max |diff| = {diff}')

## 1. Microbenchmark: how long does it take to load *m* experts?
Unbuffered reads of *m* random experts from different layers (the same random keys as the Mac, `seed 0`) with T reader threads, each followed by its copy into a VRAM slot. `ms` is the wall time for all *m*; the split into disk read and host → VRAM copy is in `micro_split`.

In [ ]:
rng = np.random.default_rng(0)
micro, split = [], []
for T in MICRO_THREADS:
    bench = tc.ExpertCache(index, 16, 'LRU', io_threads=T, device=GPU)   # T warmed-up reader threads
    for m in range(1, 9):
        for rep in range(MICRO_REPS):
            keys = [(int(rng.integers(16)), int(rng.integers(64))) for _ in range(m)]
            ms = bench.load([(k, i) for i, k in enumerate(keys)])
            micro.append({'threads': T, 'm': m, 'ms': ms})
    s = bench.summary()
    split.append({'threads': T, 'ms_per_disk_read': s['ms_per_disk_read'], 'ms_per_h2d': s['ms_per_h2d'],
                  'disk_GB_s': s['disk_GB'] / s['disk_s'] * T if s['disk_s'] else float('nan')})
    bench.close()
micro = pd.DataFrame(micro)
TABLES['micro_reads'] = micro
TABLES['micro_split'] = pd.DataFrame(split)
mt = micro.groupby(['m', 'threads']).ms.median().unstack()
display(TABLES['micro_split'].style.format(precision=2))
print('Median ms to load m experts (rows) with T threads (columns):')
mt.style.format('{:.1f}')

## 2. A MoE block that can overlap
`OverlapMoE` keeps the router and cache decisions of `tc.CachedMoE` and changes only *when* things run. Each expert's output is kept apart and added in the stock order at the end, so every mode is bit-exact.

Safety: hit computation only reads hit slots (pinned for this call); misses are written into slots freed by evictions, which no queued kernel reads (routing already waited for every earlier kernel).

In [ ]:
class OverlapMoE(tc.CachedMoE):
    mode = 'sync'

    def forward(self, x):
        shape = x.shape
        x_flat = x.reshape(-1, shape[-1])
        t0 = time.perf_counter()
        inds, scores, ind_np, sc_np, eids, mass = self.routing(x_flat)    # waits for the GPU
        t_known = time.perf_counter()
        keys = [(self.layer, int(e)) for e in eids]
        cache = self.cache
        assert len(keys) <= cache.capacity
        slots, misses = cache.plan(keys, mass)
        slot_of = dict(zip(eids.tolist(), slots))
        missed = {k[1] for k, _ in misses}
        idx = self.token_index(ind_np, eids, x_flat.device)

        def out(e):
            tok, pos = idx[e]
            return tok, self.expert_out(x_flat, slot_of[e], tok, pos, scores).to(x_flat.dtype)

        if self.mode == 'sync' or not misses or len(misses) == len(keys):
            io = cache.load(misses)
            parts = {e: out(e) for e in slot_of}
        else:
            parts = {e: out(e) for e in slot_of if e not in missed}    # GPU starts on resident experts...
            io = cache.load(misses)                                      # ...while the CPU loads the misses
            parts.update({e: out(e) for e in missed})
        y = torch.zeros_like(x_flat)
        for e in sorted(parts):                                          # stock order: exact in every mode
            tok, h = parts[e]
            y.index_add_(0, tok, h)
        tc.synchronize(x_flat.device)
        t_end = time.perf_counter()
        CALLS.append({'layer': self.layer, 'n': len(ind_np), 'm': len(misses), 'hits': len(keys) - len(misses),
                      'route_wait_ms': (t_known - t0) * 1000, 't_moe_ms': (t_end - t_known) * 1000,
                      'io_ms': io, 'phase': PHASE[0]})
        return self._finish(y, shape)

# swap the blocks in place (same router weights)
for layer in model.model.layers:
    layer.mlp.__class__ = OverlapMoE
CALLS, PHASE = [], ['prefill']
print('OverlapMoE installed on', len(model.model.layers), 'layers')

## 3. Real runs, one per mode

In [ ]:
# A small mixed request set written for this axis (one or two per kind of request).
prompt_file = rs.load_prompts(AXIS_DIR)   # axis-4-io-compute-overlap/prompts.json, shared by every device
PROMPTS = prompt_file['PROMPTS']
reqs = tc.requests(PROMPTS, n_per_domain=REQUESTS_PER_DOMAIN, order='interleave')

calls, runs, texts = [], [], {}
for mname, (mode, threads) in MODES.items():
    tc.free_memory()
    cache = tc.ExpertCache(index, K, 'LRU', io_threads=threads, device=GPU)
    OverlapMoE.mode = mode
    CALLS.clear()

    def obs_phase(ph, i):
        PHASE[0] = ph
    tc.attach_cache(model, cache, [])
    out = []
    t0 = time.perf_counter()
    dec_tok = dec_s = 0
    for r in reqs:
        g = tc.generate(model, tokenizer, tc.chat_tokens(tokenizer, r['text']), MAX_NEW_TOKENS, on_step=obs_phase)
        out.append(g.tokens); dec_tok += len(g.step_ms); dec_s += g.decode_s
    s = cache.summary()
    texts[mname] = out
    c = pd.DataFrame(CALLS); c['mode'] = mname; c['call'] = range(len(c))
    calls.append(c)
    runs.append({'mode': mname, 'decode_tok_s': dec_tok / dec_s, 'wall_s': time.perf_counter() - t0,
                 'hits': s['hits'], 'misses': s['misses'], 'io_s': s['io_ms'] / 1000,
                 'disk_s': s['disk_s'], 'h2d_s': s['h2d_s']})
    print(f'{mname:10s}: decode {runs[-1]["decode_tok_s"]:.2f} tok/s | hits {s["hits"]} misses {s["misses"]} | '
          f'load wall {s["io_ms"] / 1000:.1f}s (disk {s["disk_s"]:.1f}s, copy {s["h2d_s"]:.1f}s summed over threads)')
    cache.close(); tc.detach_cache(model)

calls = pd.concat(calls, ignore_index=True)
runs = pd.DataFrame(runs)
TABLES['runs'] = runs
# per-call timings, summarised by (mode, phase, misses in the call) for the results file
TABLES['calls_by_m'] = (calls.groupby(['mode', 'phase', 'm'], sort=False)
    .agg(calls=('t_moe_ms', 'size'), t_moe_ms=('t_moe_ms', 'median'), route_wait_ms=('route_wait_ms', 'median'),
         io_ms=('io_ms', 'median'), t_moe_mean_ms=('t_moe_ms', 'mean')).reset_index())

# same cache decisions and same output in every mode
ref = calls[calls['mode'] == 'sync-1'].m.to_numpy()
for mname in MODES:
    same = np.array_equal(calls[calls['mode'] == mname].m.to_numpy(), ref)
    agree = np.mean([np.mean(np.array(a) == np.array(b)) for a, b in zip(texts[mname], texts['sync-1'])])
    print(f'{mname:10s}: identical miss pattern: {same} | tokens equal to sync-1: {agree:.1%}')
    assert same and agree == 1.0, f'{mname} changed the cache decisions or the output'

## 4. Exposed stall vs number of simultaneous misses (decode)

In [ ]:
dec = calls[calls.phase == 'decode']
t = dec.groupby(['m', 'mode']).t_moe_ms.median().unstack()[list(MODES)]
counts = dec[dec['mode'] == 'sync-1'].groupby('m').size().rename('calls')
base = t.loc[0] if 0 in t.index else t.min()
stall = t - base
print('Median T_moe per decode call (ms), by misses m:')
display(t.join(counts).style.format('{:.2f}', subset=list(MODES)))
mask = 1 - stall['overlap-1'] / stall['sync-1']
print('Share of the sync-1 stall hidden by overlap-1, by m:')
display(mask.drop(0, errors='ignore').to_frame('hidden share').T.style.format('{:.0%}'))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for mname in MODES:
    axes[0].plot(t.index, t[mname], '-o', label=mname)
axes[0].set_xlabel('misses in the call (m)'); axes[0].set_ylabel('median ms'); axes[0].set_title('MoE layer time per decode call')
for T in mt.columns:
    axes[1].plot(mt.index, mt[T], '-o', label=f'{T} thread(s)')
axes[1].set_xlabel('experts loaded (m)'); axes[1].set_ylabel('median ms'); axes[1].set_title('Load latency for m experts (microbenchmark)')
w = dec[dec['mode'] == 'sync-1']
window = w[w.m == 0].t_moe_ms.median() if (w.m == 0).any() else np.nan
attn = w.route_wait_ms.median()
axes[2].bar(['expert compute\n(all hits)', 'attention + router\n(per layer)', 'load 1 expert', 'load 8 experts\n(1 thread)'],
            [window, attn, mt.loc[1, mt.columns[0]], mt.loc[8, mt.columns[0]]], color=['C2', 'C2', 'C3', 'C3'])
axes[2].set_ylabel('ms'); axes[2].set_title('Compute window vs load latency')
for ax in axes[:2]:
    ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

## Hypothesis check (descriptive)

In [ ]:
one = mt.loc[1, mt.columns[0]]
print(f'Compute available per layer once routing is known (all-hit expert compute): {window:.2f} ms')
print(f'Attention + router per layer (cannot overlap this layer\'s loads): {attn:.2f} ms')
print(f'Loading ONE expert: {one:.2f} ms  ->  the hit-compute window covers at most {window / one:.0%} of one miss.')
for m in [1, 2, 4, 8]:
    if m in stall.index:
        print(f'm={m}: exposed stall sync-1 {stall.loc[m, "sync-1"]:.1f} ms | sync-4 {stall.loc[m, "sync-4"]:.1f} ms | '
              f'overlap-1 {stall.loc[m, "overlap-1"]:.1f} ms | overlap-4 {stall.loc[m, "overlap-4"]:.1f} ms')
best = runs.set_index('mode').decode_tok_s
print(f'\nDecode throughput: ' + ', '.join(f'{k} {v:.2f}' for k, v in best.items()) + ' tok/s')
print('Parallel reads attack the load time itself; overlap can only hide up to the compute window.')

## Notes for reading the results

* **Within-device trends first.** These are PyTorch / BF16 / CUDA numbers. The Mac runs MLX / F16 on unified memory: compare the *shape* of the curves across devices, not absolute values as one controlled experiment.
* **No page cache to purge.** Expert reads are unbuffered (`reads` = `windows-unbuffered` in the system info), so every miss really goes to the disk. If the reader had to fall back to buffered reads it prints a warning and the mode says `buffered`.
* **K is the VRAM tier.** The timed K values are those whose bank fits in the VRAM free at load (`SAFE_K`); larger K are replay-only (hit rates exact, no timing). Close other GPU programs before a full run.

## Save results
Every table above goes into **one file for this device**: `results/windows-dgpu.json`, with the configuration, prompts and machine it ran on.
The Mac and Windows iGPU notebooks write `mac.json` / `windows-igpu.json` next to it in the same format (see `experiments/README.md`).

In [ ]:
CONFIG['SAFE_K'] = globals().get('SAFE_K')
path = rs.save(RESULTS, AXIS, DEVICE, TABLES, config=CONFIG, prompts=rs.load_prompts(AXIS_DIR),
               system=tc.system_info(GPU))
print('Saved', path.relative_to(ROOT), '| tables:', {k: len(v) for k, v in TABLES.items()})

## Cross-device figures
The report figures, drawn from every device's results file in this folder (colour = device). Only shown here;
`python -m moe.figures` writes them to `figures/` once all devices have results.

In [ ]:
figs = fg.make(AXIS, RESULTS)